# Run All Implemented Phases

This notebook is the main entry point for the implemented pipeline. It validates existing Phase 1 through Phase 7 artifacts and only reruns expensive steps when their outputs are missing, stale, or explicitly forced.

In [1]:
# Cell 1 - Set up imports and make the project package importable from notebooks.
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


Project root: E:\Paper Multiclass-Intrusion-Detection-System
Config path: E:\Paper Multiclass-Intrusion-Detection-System\configs\config.yaml


In [2]:
# Cell 2 - Load the shared YAML config and resolve the main artifact paths.
from src.data_loading import load_config, resolve_project_path

config = load_config(CONFIG_PATH)
processed_dir = resolve_project_path(config["paths"]["processed_dir"], PROJECT_ROOT)
models_dir = resolve_project_path(config["paths"]["models_dir"], PROJECT_ROOT)
metrics_dir = resolve_project_path(config["paths"]["metrics_dir"], PROJECT_ROOT)
figures_dir = resolve_project_path(config["paths"]["figures_dir"], PROJECT_ROOT)

print(f"Expected feature count: {config['data']['expected_feature_count']}")
print(f"Expected classes: {config['data']['expected_num_classes']}")


Expected feature count: 69
Expected classes: 10


In [3]:
# Cell 3 - Run Phase 1 inspection only when canonical metric outputs are missing.
from src.data_loading import inspect_dataset

phase1_outputs = [
    metrics_dir / "class_distribution.csv",
    metrics_dir / "feature_summary.csv",
    metrics_dir / "dropped_features.csv",
    metrics_dir / "quasi_constant_review.csv",
    metrics_dir / "data_inspection_report.txt",
]

if all(path.exists() for path in phase1_outputs):
    class_distribution = pd.read_csv(metrics_dir / "class_distribution.csv")
    print("Phase 1 outputs already exist; loaded class distribution from disk.")
    print(f"Rows in distribution table: {len(class_distribution)}")
else:
    phase1_report = inspect_dataset(CONFIG_PATH)
    print("Phase 1 inspection completed.")
    print(phase1_report)


Phase 1 outputs already exist; loaded class distribution from disk.
Rows in distribution table: 10


In [4]:
# Cell 4 - Run Phase 2 preprocessing only when processed arrays or the report are missing.
from src.preprocessing import preprocess_dataset

phase2_outputs = [
    processed_dir / "X_train.npy",
    processed_dir / "X_test.npy",
    processed_dir / "y_train.npy",
    processed_dir / "y_test.npy",
    processed_dir / "minmax_scaler.joblib",
    processed_dir / "label_mapping.json",
    metrics_dir / "preprocessing_report.json",
    metrics_dir / "split_class_distribution.csv",
]

FORCE_REBUILD_PHASE2 = False
if FORCE_REBUILD_PHASE2 or not all(path.exists() for path in phase2_outputs):
    phase2_report = preprocess_dataset(CONFIG_PATH)
else:
    with open(metrics_dir / "preprocessing_report.json", "r", encoding="utf-8") as file:
        phase2_report = json.load(file)

print("Phase 2 report:")
print(json.dumps(phase2_report, indent=2))


Phase 2 report:
{
  "total_rows": 5351760,
  "valid_rows": 5350583,
  "invalid_rows_dropped": 1177,
  "train_rows": 4280466,
  "test_rows": 1070117,
  "feature_count": 69,
  "test_size": 0.2,
  "random_state": 42,
  "scaler_fit_scope": "train_only",
  "max_split_percentage_delta": 4.672838196390777e-05,
  "split_distribution_path": "E:\\Paper Multiclass-Intrusion-Detection-System\\results\\metrics\\split_class_distribution.csv",
  "X_train_shape": [
    4280466,
    69
  ],
  "X_test_shape": [
    1070117,
    69
  ],
  "y_train_shape": [
    4280466
  ],
  "y_test_shape": [
    1070117
  ]
}


In [5]:
# Cell 5 - Run Phase 3 Autoencoder training only when model artifacts are missing.
phase3_outputs = [
    models_dir / "autoencoder.pt",
    models_dir / "encoder.pt",
    metrics_dir / "autoencoder_history.csv",
    metrics_dir / "autoencoder_reconstruction_error.json",
    figures_dir / "ae_loss_curve.png",
]

FORCE_RETRAIN_PHASE3 = False
phase3_needs_training = FORCE_RETRAIN_PHASE3 or not all(path.exists() for path in phase3_outputs)

if phase3_needs_training:
    from src.autoencoder import train_autoencoder

    phase3_report = train_autoencoder(CONFIG_PATH, device_name="auto")
else:
    with open(metrics_dir / "autoencoder_reconstruction_error.json", "r", encoding="utf-8") as file:
        phase3_report = json.load(file)

print("Phase 3 reconstruction report:")
print(json.dumps(phase3_report, indent=2))
print("Autoencoder checkpoint exists:", (models_dir / "autoencoder.pt").exists())
print("Encoder checkpoint exists:", (models_dir / "encoder.pt").exists())


Phase 3 reconstruction report:
{
  "train_reconstruction_mse": 0.00011258973852168794,
  "test_reconstruction_mse": 0.00011356514788172808,
  "epochs": 50,
  "batch_size": 4096,
  "device": "cpu",
  "input_dim": 69,
  "latent_dim": 16,
  "training_batch_order": "reshuffled_each_epoch",
  "model_selection": "best_validation_loss",
  "best_epoch": 48,
  "best_val_loss": 0.00010913831824187403,
  "reconstruction_scope": "full_train_array_including_internal_validation",
  "autoencoder_path": "E:\\Paper Multiclass-Intrusion-Detection-System\\models\\autoencoder.pt",
  "encoder_path": "E:\\Paper Multiclass-Intrusion-Detection-System\\models\\encoder.pt",
  "history_path": "E:\\Paper Multiclass-Intrusion-Detection-System\\results\\metrics\\autoencoder_history.csv",
  "loss_curve_path": "E:\\Paper Multiclass-Intrusion-Detection-System\\results\\figures\\ae_loss_curve.png"
}
Autoencoder checkpoint exists: True
Encoder checkpoint exists: True


In [6]:
# Cell 6 - Run Phase 4 latent extraction only when latent artifacts are missing.
phase4_outputs = [
    processed_dir / "Z_train.npy",
    processed_dir / "Z_test.npy",
    metrics_dir / "latent_extraction_report.json",
]

FORCE_REEXTRACT_PHASE4 = False
phase4_needs_extraction = FORCE_REEXTRACT_PHASE4 or not all(path.exists() for path in phase4_outputs)

if phase4_needs_extraction:
    from src.latent_extraction import extract_latent_features

    phase4_report = extract_latent_features(CONFIG_PATH, device_name="auto")
else:
    with open(metrics_dir / "latent_extraction_report.json", "r", encoding="utf-8") as file:
        phase4_report = json.load(file)

print("Phase 4 latent extraction report:")
print(json.dumps(phase4_report, indent=2))


Phase 4 latent extraction report:
{
  "input_dim": 69,
  "latent_dim": 16,
  "batch_size": 4096,
  "device": "cpu",
  "encoder_path": "E:\\Paper Multiclass-Intrusion-Detection-System\\models\\encoder.pt",
  "Z_train": {
    "path": "E:\\Paper Multiclass-Intrusion-Detection-System\\data\\processed\\Z_train.npy",
    "shape": [
      4280466,
      16
    ],
    "dtype": "float32",
    "finite": true,
    "min": 0.0,
    "max": 19.60346221923828
  },
  "Z_test": {
    "path": "E:\\Paper Multiclass-Intrusion-Detection-System\\data\\processed\\Z_test.npy",
    "shape": [
      1070117,
      16
    ],
    "dtype": "float32",
    "finite": true,
    "min": 0.0,
    "max": 23.2085018157959
  }
}


In [7]:
# Cell 7 - Prepare or validate all Phase 5 training-only imbalance scenarios.
from src.imbalance import prepare_all_imbalance_scenarios

FORCE_REBUILD_PHASE5 = False
phase5_reports = prepare_all_imbalance_scenarios(
    CONFIG_PATH,
    force=FORCE_REBUILD_PHASE5,
)
phase5_summary = pd.DataFrame([
    {
        "scenario": report["scenario"],
        "input_rows": report["input_rows"],
        "output_rows": report["output_rows"],
    }
    for report in phase5_reports.values()
])
phase5_summary


,scenario,input_rows,output_rows
0,s1_none,4280466,4280466
1,s2_class_weight,4280466,4280466
2,s3_upsampling,4280466,20112470
3,s4_downsampling,4280466,1160


In [8]:
# Cell 8 - Train or reuse all Phase 6 LightGBM models and test predictions.
from src.classifier import train_all_scenarios

FORCE_RETRAIN_PHASE6 = False
phase6_reports = train_all_scenarios(CONFIG_PATH, force=FORCE_RETRAIN_PHASE6)
phase6_summary = pd.DataFrame([
    {
        "scenario": report["scenario"],
        "training_rows": report["training_rows"],
        "test_rows": report["test_rows"],
        "all_classes_predicted": report["all_configured_classes_predicted"],
        "reused": report.get("reused_existing_artifacts", False),
    }
    for report in phase6_reports.values()
])
phase6_summary


,scenario,training_rows,test_rows,all_classes_predicted,reused
0,s1_none,4280466,1070117,True,True
1,s2_class_weight,4280466,1070117,True,True
2,s3_upsampling,20112470,1070117,True,True
3,s4_downsampling,1160,1070117,True,True


In [9]:
# Cell 9 - Evaluate all Phase 6 predictions and regenerate Phase 7 reports.
from src.evaluation import evaluate_all_scenarios

phase7_result = evaluate_all_scenarios(CONFIG_PATH)
phase7_result["summary"]


,scenario,accuracy,macro_precision,macro_recall,macro_f1,training_seconds,prediction_seconds
0,s1_none,0.843600,0.320398,0.337571,0.314168,104.975497,10.831626
1,s2_class_weight,0.728500,0.410582,0.562000,0.424949,104.912774,10.577099
2,s3_upsampling,0.727819,0.410291,0.557990,0.423284,532.179907,12.209939
3,s4_downsampling,0.615504,0.305047,0.490024,0.292287,1.636508,13.964128


In [10]:
# Cell 10 - Summarize the implemented pipeline artifacts in one table.
summary_rows = [
    {"phase": "Phase 1", "artifact": "class_distribution.csv", "exists": (metrics_dir / "class_distribution.csv").exists()},
    {"phase": "Phase 1", "artifact": "feature_summary.csv", "exists": (metrics_dir / "feature_summary.csv").exists()},
    {"phase": "Phase 2", "artifact": "X_train.npy", "exists": (processed_dir / "X_train.npy").exists()},
    {"phase": "Phase 2", "artifact": "X_test.npy", "exists": (processed_dir / "X_test.npy").exists()},
    {"phase": "Phase 3", "artifact": "autoencoder.pt", "exists": (models_dir / "autoencoder.pt").exists()},
    {"phase": "Phase 3", "artifact": "encoder.pt", "exists": (models_dir / "encoder.pt").exists()},
    {"phase": "Phase 3", "artifact": "ae_loss_curve.png", "exists": (figures_dir / "ae_loss_curve.png").exists()},
    {"phase": "Phase 4", "artifact": "Z_train.npy", "exists": (processed_dir / "Z_train.npy").exists()},
    {"phase": "Phase 4", "artifact": "Z_test.npy", "exists": (processed_dir / "Z_test.npy").exists()},
    {"phase": "Phase 4", "artifact": "latent_extraction_report.json", "exists": (metrics_dir / "latent_extraction_report.json").exists()},
    {"phase": "Phase 7", "artifact": "summary_comparison.csv", "exists": (metrics_dir / "summary_comparison.csv").exists()},
    {"phase": "Phase 7", "artifact": "phase7_analysis.md", "exists": (metrics_dir / "phase7_analysis.md").exists()},
    {"phase": "Phase 7", "artifact": "metrics_comparison.png", "exists": (figures_dir / "metrics_comparison.png").exists()},
]
for scenario in config["imbalance"]["scenarios"]:
    summary_rows.extend([
        {"phase": "Phase 5", "artifact": f"imbalance_report_{scenario}.json", "exists": (metrics_dir / f"imbalance_report_{scenario}.json").exists()},
        {"phase": "Phase 6", "artifact": f"lgbm_{scenario}.txt", "exists": (models_dir / f"lgbm_{scenario}.txt").exists()},
        {"phase": "Phase 6", "artifact": f"y_pred_{scenario}.npy", "exists": (processed_dir / f"y_pred_{scenario}.npy").exists()},
    ])
pd.DataFrame(summary_rows)


,phase,artifact,exists
0,Phase 1,class_distribution.csv,True
1,Phase 1,feature_summary.csv,True
2,Phase 2,X_train.npy,True
3,Phase 2,X_test.npy,True
4,Phase 3,autoencoder.pt,True
5,Phase 3,encoder.pt,True
6,Phase 3,ae_loss_curve.png,True
7,Phase 4,Z_train.npy,True
8,Phase 4,Z_test.npy,True
9,Phase 4,latent_extraction_report.json,True
